In [0]:
jdbc_url = (
    "jdbc:postgresql://**********ooler.c-9.us-east-1.aws.neon.tech:5432/neondb?sslmode=require"
)

In [0]:
df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "sales")
    .option("user", "neondb_owner")
    .option("password", "***")
    .load()
)

In [0]:
display(df)

customer_id,amount,region,updated_at
1,100.0,Europe,2026-06-18T15:28:32.487Z
2,200.0,Asia,2026-06-18T15:28:32.487Z
3,300.0,Europe,2026-06-18T15:28:32.487Z
4,400.0,Germany,2026-06-18T15:28:32.487Z


In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "ingestion_time",
    current_timestamp()
)

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("bronze.sales")

In [0]:
max_ts = (
    df.selectExpr(
        "max(updated_at) as max_ts"
    )
    .collect()[0]["max_ts"]
)

In [0]:
spark.sql(f"""

UPDATE metadata.pipeline_metadata

SET watermark =
'{max_ts}'

WHERE pipeline_name =
'sales_pipeline'

""")

DataFrame[num_affected_rows: bigint]